# Web API 
- 웹(HTTP)을 통해 프로그램과 프로그램이 데이터를 주고받거나 특정 기능을 사용할 수 있도록 만들어 놓은 규칙이다.
    - 네이버 API, 공공데이터 API, Google API, Gmail API 등

## 야후 파이낸스(Yahoo Finance)

In [24]:
# !pip install yfinance

- [프롬프트]
   - 원유 가격 알려주는 파이썬 코드 만들어줘
   - 참고) 야후 파이낸스(Yahoo Finance)

In [25]:
import yfinance as yf

# WTI 원유 선물
oil = yf.Ticker("CL=F")

# 최근 1일 데이터
df = oil.history(period="1d", interval="1m")

# 가장 최근 가격
price = df["Close"].iloc[-1]

print(f"현재 WTI 원유 가격: ${price:.2f} / 배럴")

현재 WTI 원유 가격: $78.84 / 배럴


- [프롬프트]
   - 금가격 알려주는 파이썬 코드 만들어줘
   - 참고) 야후 파이낸스(Yahoo Finance)

In [26]:
import yfinance as yf

# 금 선물
gold = yf.Ticker("GC=F")

# 최근 데이터
df = gold.history(period="1d")

# 가장 최근 종가
price = df["Close"].iloc[-1]

print(f"현재 금 가격: ${price:.2f} / 트로이온스")

현재 금 가격: $4393.60 / 트로이온스


## 공공 포털 사이트
- https://www.data.go.kr/

- [프롬프트]
    - 1. 아래 조건을 반영해서 기상청 API 단기예보를 조회하는 파이썬 코드를 작성해줘.
        - 1-1) URL: http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst
        - 1-2) 인증키: "wdkT93ooCcYPPWCPWIluRHhUDdOeVrWNekh%2B%2BlQ5ctI5Z9zLH%2FJjV6cblkDmzlqOicnFBWwV5txBotvhrQyrjg%3D%3D"
            - 1-2-1) 이 인증키는 urllib.parse.unquote를 사용해 디코딩해서 requests의 params로 넘겨줘
        - 1-3) 파라미터는 numOfRows, pageNo, base_date, base_time, nx, ny를 딕셔너리 형태로 포함해줘.

In [11]:
import requests
from urllib.parse import unquote


# --------------------------------------------------
# 1. API URL
# --------------------------------------------------
url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"


# --------------------------------------------------
# 2. 인증키
#    URL Encoding 된 인증키를 디코딩
# --------------------------------------------------
encoded_key = (
    "wdkT93ooCcYPPWCPWIluRHhUDdOeVrWNekh%2B%2BlQ5ctI5Z9zLH%2FJjV6cblkDmzlqOicnFBWwV5txBotvhrQyrjg%3D%3D"
)

service_key = unquote(encoded_key)


# --------------------------------------------------
# 3. API 요청 파라미터
# --------------------------------------------------
params = {
    "serviceKey": service_key,
    "numOfRows": 10,
    "pageNo": 1,
    "base_date": "20260819",
    "base_time": "0600",
    "nx": 55,
    "ny": 127,
    "dataType": "JSON"
}


# --------------------------------------------------
# 4. API 호출
# --------------------------------------------------
response = requests.get(
    url,
    params=params,
    timeout=10
)


# --------------------------------------------------
# 5. 결과 확인
# --------------------------------------------------
print("HTTP 상태코드 :", response.status_code)

if response.status_code == 200:

    data = response.json()

    print("결과코드 :", data["response"]["header"]["resultCode"])
    print("결과메시지 :", data["response"]["header"]["resultMsg"])

    # 실제 날씨 데이터
    items = data["response"]["body"]["items"]["item"]

    print("\n===== 날씨 정보 =====")

    for item in items:
        print(
            f"카테고리 : {item['category']}, "
            f"발표일자 : {item['baseDate']}, "
            f"발표시간 : {item['baseTime']}, "
            f"값 : {item['obsrValue']}"
        )

else:
    print("API 호출 실패")
    print(response.text)

HTTP 상태코드 : 200
결과코드 : 00
결과메시지 : NORMAL_SERVICE

===== 날씨 정보 =====
카테고리 : PTY, 발표일자 : 20260819, 발표시간 : 0600, 값 : 0
카테고리 : REH, 발표일자 : 20260819, 발표시간 : 0600, 값 : 94
카테고리 : RN1, 발표일자 : 20260819, 발표시간 : 0600, 값 : 0
카테고리 : T1H, 발표일자 : 20260819, 발표시간 : 0600, 값 : 25.6
카테고리 : UUU, 발표일자 : 20260819, 발표시간 : 0600, 값 : 0.1
카테고리 : VEC, 발표일자 : 20260819, 발표시간 : 0600, 값 : 225
카테고리 : VVV, 발표일자 : 20260819, 발표시간 : 0600, 값 : 0.1
카테고리 : WSD, 발표일자 : 20260819, 발표시간 : 0600, 값 : 0.1


# iframe
- 1. Python과 Selenium을 사용하여 아래의 조건에 맞게 코드를 작성해줘
    - 1-1) iframe 요소에 아이디는 "_IframeBannerRight_tgtLREC"이며, 이를 i_frame 변수에 할당해줘
    - 1-2) i_frame 할당된 요소안 새로운 html 문서안 요소중 class명은 "content_info"이며, 이를 content 변수에 할당해줘
    - 1-3) content 변수에 할당된 요소안에는 각기 다른 자식 태그가 3개 있는데, 자식 태그의 내용을 출력해줘

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# 브라우저 실행
driver = webdriver.Chrome()

# 페이지 접속
driver.get("https://finance.naver.com/item/main.naver?code=005930")

time.sleep(2)

# 1-1) iframe 요소를 찾아 i_fream 변수에 할당
i_frame = driver.find_element(
    By.CSS_SELECTOR,
    "#_IframeBannerRight_tgtLREC"
)

# iframe 안으로 전환
driver.switch_to.frame(i_frame)

# 1-2) iframe 내부의 새로운 HTML 문서에서
#      class명이 content_info인 요소를 content 변수에 할당
content = driver.find_element(
    By.CLASS_NAME,
    "content_info"
)

# 1-3) content 안에 있는 자식 태그 3개의 내용 출력
children = content.find_elements(
    By.XPATH,
    "./*"
)

for child in children:
    print(child.text)

# 작업이 끝났으면 원래 HTML 문서로 돌아오기
driver.switch_to.default_content()

104·113㎡ 6억대 하이엔드아파트
49층 오션뷰와 수영장, 3천만원대로 입주 시까지
청라하늘대교 디에트르라메르


In [ ]:
# End Of -------------------------------------------------------------------------------------